# Лабораторная работа №4

#### Задание

Признак $X$ распределен в генеральной совокупности нормально с известным средним квадратическим отклонением $\sigma$. Данные выборочного наблюдения представлены вариационным рядом.

По данным выборки требуется:

1) найти доверительный интервал для математического ожидания генеральной совокупности с надежностью $0.99$;
2) составить функцию плотности распределения вероятности для предполагаемого нормального закона распределения;
3) определить надежность интервальной оценки $(\alpha;\ \beta)$ математического ожидания;
4) вычислить точечную оценку среднего квадратического отклонения двумя различными способами;
5) считая среднее квадратическое отклонение генеральной совокупности неизвестным, найти для него доверительный интервал с надежностью $0.95$;
6) определить каким был объем выборки, если доверительный интервал $(0;\ 2.33S)$ для среднего квадратического отклонения генеральной совокупности ожидается с надежностью $0.999$.

*[Здесь тоже подлянка: в примере доверительный интервал в 6 задании $(0; 3S)$, а не $(0; 2.33S)$]*

In [326]:
# @title
import numpy as np
from IPython.display import display, Markdown
from scipy import stats   # для t_gamma (распределение Стьюдента) и t из Ф(t)
import math as math

# ─── Данные вариантов 1–8 ─────────────────────────────────────────────
variant_data = {
    0: {
        'left':  [11, 13, 15, 17, 19, 21, 23],
        'right': [13, 15, 17, 19, 21, 23, 25],
        'freq':  [2, 7, 15, 18, 12, 5, 1],
        'sigma': 3,
        'alpha': 17.5,
        'beta':  17.9
    },
    1: {
        'left':  [14.5, 19.5, 24.5, 29.5, 34.5, 39.5, 44.5, 49.5, 54.5],
        'right': [19.5, 24.5, 29.5, 34.5, 39.5, 44.5, 49.5, 54.5, 59.5],
        'freq':  [2, 4, 11, 12, 18, 11, 10, 8, 4],
        'sigma': 10,
        'alpha': 29,
        'beta':  47
    },
    2: {
        'left':  [45.5, 50.5, 55.5, 60.5, 65.5, 70.5, 75.5, 80.5, 85.5],
        'right': [50.5, 55.5, 60.5, 65.5, 70.5, 75.5, 80.5, 85.5, 90.5],
        'freq':  [1, 4, 10, 12, 18, 11, 10, 8, 4],
        'sigma': 32,
        'alpha': 60,
        'beta':  78
    },
    3: {
        'left':  [8, 15, 21, 27, 33, 39, 45, 51, 57],
        'right': [14, 21, 27, 33, 39, 45, 51, 57, 63],
        'freq':  [3, 4, 10, 12, 18, 11, 10, 7, 4],
        'sigma': 12,
        'alpha': 21,
        'beta':  52
    },
    4: {
        'left':  [88, 94, 100, 106, 112, 118, 124, 130, 136],
        'right': [94, 100, 106, 112, 118, 124, 130, 136, 142],
        'freq':  [4, 4, 10, 12, 18, 11, 10, 7, 4],
        'sigma': 78,
        'alpha': 100,
        'beta':  130
    },
    5: {   # ВНИМАНИЕ: в варианте 5 опечатка — интервалы не монотонно возрастают
        'left':  [81, 87, 93, 99, 105, 101, 107, 113, 119],
        'right': [87, 93, 99, 105, 111, 107, 113, 119, 125],
        'freq':  [3, 4, 10, 13, 17, 12, 10, 7, 4],
        'sigma': 67,
        'alpha': 94,
        'beta':  114
    },
    6: {
        'left':  [17.5, 23.5, 29.5, 35.5, 41.5, 47.5, 53.5, 59.5, 65.5],
        'right': [23.5, 29.5, 35.5, 41.5, 47.5, 53.5, 59.5, 65.5, 71.5],
        'freq':  [2, 5, 10, 13, 16, 12, 11, 7, 4],
        'sigma': 14,
        'alpha': 40,
        'beta':  52
    },
    7: {
        'left':  [19, 22, 25, 28, 31, 34, 37, 40, 43],
        'right': [22, 25, 28, 31, 34, 37, 40, 43, 46],
        'freq':  [2, 5, 6, 9, 12, 10, 8, 5, 1],
        'sigma': 9,
        'alpha': 30,
        'beta':  35
    },
    8: {
        'left':  [52, 55, 58, 61, 64, 67, 70, 73, 76],
        'right': [55, 58, 61, 64, 67, 70, 73, 76, 79],
        'freq':  [3, 5, 6, 9, 12, 10, 8, 5, 1],
        'sigma': 30,
        'alpha': 60,
        'beta':  70
    }
}

# ВНИМАНИЕ: в варианте 5 опечатка — интервалы не монотонно возрастают
variant = 8  # выберите номер своего варианта (1--8)

data = variant_data[variant]
left = np.array(data['left'])
right = np.array(data['right'])
freq = np.array(data['freq'])
sigma = data['sigma']
alpha = data['alpha']
beta = data['beta']

nsize = freq.sum()
nnn = len(left)



bounds_strings = []

counts_strings = []

for i in range (0, nnn):
  bstring = f"{left[i]}$-${right[i]}"
  bounds_strings.append(bstring)
  counts_strings.append(str(freq[i]))


# Объединяем данные в список строк: каждая строка = название + значения
data_rows = [ ["$x_i;x_{i+1}$"] + bounds_strings, ["$n_i$"] + counts_strings]


# Заголовки столбцов (первая колонка — названия строк)
headers = [' '] * len(data_rows[0])

# Начало таблицы
table = "| " + " | ".join(headers) + " |\n"
table += "|" + "|".join(["---"] * len(headers)) + "|\n"

# Перебираем строки данных и добавляем в таблицу
for row in data_rows:
    # Форматируем ячейки (если число — можно указать количество знаков)
    param = row[0]
    val1 = row[1]
    val2 = row[2]
    table += "| " + " | ".join(row) + " |\n"

# Красивый вывод
display(Markdown(table))

hhh = ""
if(variant == 0):
  hhh = "Выберите свой вариант"

s = fr"""
####Таблица 1 $-$ вариационный ряд варианта $\textbf{variant}$. $\sigma = \textbf{{{sigma}}}$, $\alpha = \textbf{{{alpha}}}$, $\beta = \textbf{{{beta}}}$. {hhh}
"""
display(Markdown(s))

|   |   |   |   |   |   |   |   |   |   |
|---|---|---|---|---|---|---|---|---|---|
| $x_i;x_{i+1}$ | 52$-$55 | 55$-$58 | 58$-$61 | 61$-$64 | 64$-$67 | 67$-$70 | 70$-$73 | 73$-$76 | 76$-$79 |
| $n_i$ | 3 | 5 | 6 | 9 | 12 | 10 | 8 | 5 | 1 |



####Таблица 1 $-$ вариационный ряд варианта $\textbf8$. $\sigma = \textbf{30}$, $\alpha = \textbf{60}$, $\beta = \textbf{70}$. 


####Решение


In [318]:
# @title
cdot = "⋅"
plusstring = "+"

xb_string = ""

centre = []

for i in range (0, nnn):
  centre.append(round ( (right[i] + left[i]) / 2 , 1))



for i in range (nnn - 1):
  xb_string = xb_string + str(centre[i]) + cdot + str(freq[i]) + " " + plusstring + " "
xb_string = xb_string + str(centre[nnn-1]) + cdot + str(freq[nnn-1])

xb = sum(np.array(centre) * np.array(freq)) / nsize
xb_rounded = round(xb, 1)



s1 = fr"""
Вычислим середины интервалов $x_i^{{*}}$ и, по ним, среднюю арифметическую:
"""
s2 = fr"""
$$ \overline {{ x }}_{{B}}  = \frac{1}{{n}} \sum_{{i = 1}}^{{k}}  x_{{i}}^{{*}} n_{{i}} = \frac{1}{{ {nsize} }} \cdot ( {xb_string} ) \approx \textbf{{ {xb_rounded} }} $$
"""

display(Markdown(s1))
display(Markdown(s2))


Вычислим середины интервалов $x_i^{*}$ и, по ним, среднюю арифметическую:



$$ \overline { x }_{B}  = \frac1{n} \sum_{i = 1}^{k}  x_{i}^{*} n_{i} = \frac1{ 59 } \cdot ( 53.5⋅3 + 56.5⋅5 + 59.5⋅6 + 62.5⋅9 + 65.5⋅12 + 68.5⋅10 + 71.5⋅8 + 74.5⋅5 + 77.5⋅1 ) \approx \textbf{ 65.3 } $$


In [319]:
# @title




s = fr"""
Формула для функции плотности вероятности $f(x) = \frac{{1}}{{\sigma \sqrt{{2 \pi}} }} \cdot \mathrm{{e}}^{{\frac{{(x - a)^2}}{{2\sigma^2}}  }}$ в нашем случае примет вид $f(x) = \frac{{1}}{{ {sigma} \sqrt{{2 \pi}} }} \cdot \mathrm{{e}}^{{\frac{{(x - {xb_rounded})^2}}{{ { round(2* (sigma**2), 1)  }  }}  }}$
"""
display(Markdown(s))



Формула для функции плотности вероятности $f(x) = \frac{1}{\sigma \sqrt{2 \pi} } \cdot \mathrm{e}^{\frac{(x - a)^2}{2\sigma^2}  }$ в нашем случае примет вид $f(x) = \frac{1}{ 30 \sqrt{2 \pi} } \cdot \mathrm{e}^{\frac{(x - 65.3)^2}{ 1800  }  }$


*[Если плохо видно, добавьте перед словом "Формула" в коде ### или ##]*

Так как $\sigma$ известно, доверительный интервал для математического ожидания найдём по формуле:

$$
\left( \bar{x}_B - t_\gamma \frac{\sigma}{\sqrt{n}};\;
       \bar{x}_B + t_\gamma \frac{\sigma}{\sqrt{n}} \right),
$$

$t_\gamma$ найдём из соотношения $2\Phi(t)=0.99$. $\Phi(t)=0.495$. По таблице приложения $2$: $t_\gamma = 2.58$.  


In [320]:
# @title
t_gamma = 2.58


bigfrac = (t_gamma * sigma)/math.sqrt(nsize)

calculated_alpha = xb_rounded - bigfrac

calculated_beta = xb_rounded + bigfrac

s=fr"""
Искомый доверительный интервал (с надёжностью $\gamma = 0.99$): $\left(   {xb_rounded} - \frac{{ {t_gamma} \cdot {sigma} }} {{ \sqrt {{{nsize}}} }};  {xb_rounded} + \frac{{ {t_gamma} \cdot {sigma} }} {{ \sqrt {{{nsize}}} }} \right) $   или $ \left(  {{{round(calculated_alpha ,1)}}}; {{{round(calculated_beta ,1)}}} \right)$
"""
display(Markdown(s))


Искомый доверительный интервал (с надёжностью $\gamma = 0.99$): $\left(   65.3 - \frac{ 2.58 \cdot 30 } { \sqrt {59} };  65.3 + \frac{ 2.58 \cdot 30 } { \sqrt {59} } \right) $   или $ \left(  {55.2}; {75.4} \right)$


In [321]:
# @title

eps = (beta - alpha) / 2

calculated_t = eps * math.sqrt(nsize) / sigma


s = fr"""

Найдём надёжность интервальной оценки $({alpha}; {beta})$ математического ожидания генеральной совокупности. Так как средняя арифметическая является центром доверительного интервала, точность оценки $\varepsilon$ составит $\varepsilon = \frac{{ {beta} - {alpha} }}{{2}} = {round(eps, 2)}$. С другой стороны $\varepsilon = \frac{{t_\gamma \cdot \sigma}}{{\sqrt n}}$. Получим уравнение: ${round(eps, 2)} = \frac{{t_\gamma \cdot {sigma}}} {{ \sqrt{{{nsize}}}   }}$. Отсюда $t_\gamma = \textbf{{{round(calculated_t, 2)}}}$

"""

display (Markdown(s))








Найдём надёжность интервальной оценки $(60; 70)$ математического ожидания генеральной совокупности. Так как средняя арифметическая является центром доверительного интервала, точность оценки $\varepsilon$ составит $\varepsilon = \frac{ 70 - 60 }{2} = 5.0$. С другой стороны $\varepsilon = \frac{t_\gamma \cdot \sigma}{\sqrt n}$. Получим уравнение: $5.0 = \frac{t_\gamma \cdot 30} { \sqrt{59}   }$. Отсюда $t_\gamma = \textbf{1.28}$



In [322]:
# @title
import scipy.stats as stats
calculated_gamma = stats.norm.cdf(calculated_t) - 0.5

s = fr"""
По таблице приложения $2$: $\Phi({round(calculated_t, 2)}) = \textbf{{{ round(calculated_gamma, 4)}}}$. Тогда $\gamma =
2\Phi(t) = 2\Phi({round(calculated_t, 2)}) = \textbf{{{round(calculated_gamma*2, 4)}}}$. То есть с вероятностью $\textbf{{{round(calculated_gamma*2, 4)}}}$ можно ожидать, что математическое ожидание генеральной совокупности попадает в интервал $ \left(  {alpha}; {beta} \right) $
"""

display(Markdown(s))


По таблице приложения $2$: $\Phi(1.28) = \textbf{0.3998}$. Тогда $\gamma = 
2\Phi(t) = 2\Phi(1.28) = \textbf{0.7995}$. То есть с вероятностью $\textbf{0.7995}$ можно ожидать, что математическое ожидание генеральной совокупности попадает в интервал $ \left(  60; 70 \right) $


Несмещенной точечной оценкой среднего квадратического отклонения является исправленное среднее квадратическое отклонение. Вычислим исправленое среднее квадратическое отклонение двумя способами:

In [323]:
# @title
mid = np.array(centre)
x_b = xb
n = nsize
dev = [mid[i] - x_b for i in range(len(mid))]
dev = mid - x_b

ni_dev_sq = [freq[i] * dev[i]**2 for i in range(len(freq))]
ni_dev_sq = freq * (dev**2)

sum_ni_dev_sq = sum(ni_dev_sq)
ni_mid_sq = [freq[i] * mid[i]**2 for i in range(len(freq))]
sum_ni_mid_sq = sum(ni_mid_sq)
S_first = np.sqrt(sum_ni_dev_sq / (n - 1))
S_second = np.sqrt((sum_ni_mid_sq - n * x_b**2) / (n - 1))

s = f"""

Первый способ: используем отклонения середин интервалов от выборочной средней.
$$ S = \\sqrt{{ \\frac{1}{{n - 1}} \\cdot \\sum n_i (x_i^* - \\overline{{x}}_B)^2 }} = \\sqrt{{\\frac{{{1}}}{{{n-1}}} \\cdot {{{round(sum_ni_dev_sq, 2)}}} }} \\approx \\textbf{{{S_first:.2f}}}$$

Второй способ: используем преобразованную формулу.
$$ S = \\sqrt{{ \\frac{{n}}{{n - 1}} \\cdot \\left( \\frac{{1}}{{n}} \\sum \\left[ n_i (x_i^*)^2 \\right] - \\overline{{x}}_B^2\\right) }} = \\sqrt{{ \\frac{{ { round(sum_ni_mid_sq - n * x_b**2, 2)} }}{{ {n - 1} }}   }} \\approx \\textbf{{{S_first:.2f}}} $$

Оба способа дают одинаковый результат: $S \\approx \\textbf{{{S_second:.2f}}}$.
"""
display(Markdown(s))



Первый способ: используем отклонения середин интервалов от выборочной средней.
$$ S = \sqrt{ \frac1{n - 1} \cdot \sum n_i (x_i^* - \overline{x}_B)^2 } = \sqrt{\frac{1}{58} \cdot {2059.63} } \approx \textbf{5.96}$$

Второй способ: используем преобразованную формулу.
$$ S = \sqrt{ \frac{n}{n - 1} \cdot \left( \frac{1}{n} \sum \left[ n_i (x_i^*)^2 \right] - \overline{x}_B^2\right) } = \sqrt{ \frac{ 2059.63 }{ 58 }   } \approx \textbf{5.96} $$

Оба способа дают одинаковый результат: $S \approx \textbf{5.96}$.


*[Здесь исправил ошибку, в первом способе при суммировании надо также умножать на частоты $n_i$: $S \neq \sqrt{{ \frac{1}{{n - 1}} \cdot \sum (x_i^* - \overline{{x}}_B)^2 }}$; $S = \sqrt{{ \frac{1}{{n - 1}} \cdot \sum \mathbf{n_i}(x_i^* - \overline{{x}}_B)^2 }}$]*

*[А во втором способе в тексте были неправильно расставлены скобки]*


In [324]:
# @title
from scipy.optimize import fsolve

gamma5 = 0.95
alpha5 = 1 - gamma5
chi2_lower = stats.chi2.ppf(alpha5 / 2, n - 1)
chi2_upper = stats.chi2.ppf(1 - alpha5 / 2, n - 1)
lower_bound = S_second * np.sqrt((n - 1) / chi2_upper)
upper_bound = S_second * np.sqrt((n - 1) / chi2_lower)




def calculate_q(n, gamma):
    """
    Вычисляет параметр q для доверительного интервала
    среднего квадратического отклонения.
    n - размер выборки
    gamma - доверительная вероятность (надежность), например 0.95
    """
    df = n - 1  # число степеней свободы

    # Функция для нахождения корня уравнения
    def equations(q):
        if q < 1:
            # Условие: P( s(1-q) < sigma < s(1+q) ) = gamma
            # Переходим к квантилям хи-квадрат
            prob = stats.chi2.cdf(df / (1 - q)**2, df) - stats.chi2.cdf(df / (1 + q)**2, df)
            return prob - gamma
        else:
            # Если q >= 1, левая граница интервала равна 0.
            # Условие превращается в: P( 0 < sigma < s(1+q) ) = gamma
            prob = stats.chi2.cdf(df / (1 + q)**2, df)
            return prob - (1 - gamma)

    # Начальное приближение зависит от размера выборки
    initial_guess = 0.5 if n > 10 else 1.5

    # Численное решение уравнения
    q_value = fsolve(equations, x0=initial_guess)[0]
    return round(q_value, 3)


qval = calculate_q(n, gamma5)

qleft = S_first - S_first * qval
qright = S_first + S_first * qval

sg = "<"

if (qval >= 1):
  sg = ">"




s = fr"""
Пусть среднее квадратическое отклонение генеральной совокупности неизвестно. По таблице приложения 5: $q = q({gamma5}, {n}) = \textbf{{{qval}}} {sg} 1$
"""

sa=fr"""
Поэтому доверительный интервал для неизвестного среднего квадратического отклонения будем искать по формуле $\left( S - Sq; S + sq \right)$: $\left( {round(S_first, 2)} - {round(S_first, 2)} \cdot {qval}; {round(S_first, 2)} + {round(S_first, 2)} \cdot {qval} \right)$ или $\left( {round(qleft, 2)} ; {round(qright, 2)}\right)$
"""

ss=fr"""
Поэтому доверительный интервал для неизвестного среднего квадратического отклонения будем искать по формуле $\left( 0; S + sq \right)$: $\left( {0} ; {round(S_first, 2)} + {round(S_first, 2)} \cdot {qval} \right)$ или $\left( 0; {round(qright, 2)}  \right)$
"""



display(Markdown(s))

if(qval >= 1):
  display(Markdown(ss))
else:
  display (Markdown(sa))



Пусть среднее квадратическое отклонение генеральной совокупности неизвестно. По таблице приложения 5: $q = q(0.95, 59) = \textbf{0.19} < 1$



Поэтому доверительный интервал для неизвестного среднего квадратического отклонения будем искать по формуле $\left( S - Sq; S + sq \right)$: $\left( 5.96 - 5.96 \cdot 0.19; 5.96 + 5.96 \cdot 0.19 \right)$ или $\left( 4.83 ; 7.09\right)$


In [325]:
# @title
gamma = 0.999

k = 2.33
if(variant == 0):
  k = 3

k2 = k - 1.0

def find_sample_size(q, gamma):
    """
    Находит минимальный размер выборки n по известным q и gamma.
    q - коэффициент точности
    gamma - доверительная вероятность (например, 0.95)
    """
    # Задаем диапазон перебора выборки (от 2 до 10 000)
    for n in range(2, 10000):
        df = n - 1  # число степеней свободы

        if q < 1:
            # Считаем вероятность попадания в интервал для текущего n
            prob = stats.chi2.cdf(df / (1 - q)**2, df) - stats.chi2.cdf(df / (1 + q)**2, df)
            # Как только расчетная вероятность покрывает gamma, n найдено
            if prob >= gamma:
                return n
        else:
            # Для q >= 1 левая граница интервала равна 0
            prob = stats.chi2.cdf(df / (1 + q)**2, df)
            if prob <= (1 - gamma):
                return n

    return None

calculated_n = find_sample_size(k2, gamma)


s = fr"""
Определим, каким был объем выборки, если $(0, {k}S)$ $-$ доверительный интервал, покрывающий среднее квадратическое отклонение генеральной совокупности с доверительной вероятностью $0.999$. Правая граница доверительного интервала $S + Sq = {k}S$, $q = {k2}$. По таблице приложения 5 для $q = {k2}$ и $\gamma = {gamma}$ находим объём выборки: $n = {calculated_n}$
"""
display(Markdown(s))



Определим, каким был объем выборки, если $(0, 2.33S)$ $-$ доверительный интервал, покрывающий среднее квадратическое отклонение генеральной совокупности с доверительной вероятностью $0.999$. Правая граница доверительного интервала $S + Sq = 2.33S$, $q = 1.33$. По таблице приложения 5 для $q = 1.33$ и $\gamma = 0.999$ находим объём выборки: $n = 13$


*[В нулевом варианте ответ отличается из-за округления]*